In [1]:
# -----------------------------
# PlantDocBot: Full Text Preprocessing + BERT DataLoader Pipeline
# -----------------------------

import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# -----------------------------
# Step 1: Load Dataset
# -----------------------------
dataset_path = r"C:\Users\Sreelakshmi\OneDrive\Desktop\PlantDocBot_project\plant_data.csv"

try:
    df = pd.read_csv(dataset_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

print("Columns in dataset:", df.columns.tolist())

# -----------------------------
# Step 2: Select Text and Label Columns
# -----------------------------
text_col = 'symptom_text'
label_col = 'label'

# Drop rows with missing text or label
df = df.dropna(subset=[text_col, label_col])

# Convert to string and clean
df['text'] = df[text_col].astype(str).str.lower().str.strip()
df['label'] = df[label_col].astype(str).str.strip()

# -----------------------------
# Step 3: Label Encoding
# -----------------------------
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

# -----------------------------
# Step 4: Train / Validation / Test Split
# -----------------------------
# NOTE: stratify cannot be used because each class has only 1 sample
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(),
    df['label_encoded'].tolist(),
    test_size=0.3,  # 30% for validation + test
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,  # Split equally between val and test
    random_state=42
)

print(f"Train size: {len(train_texts)}, Val size: {len(val_texts)}, Test size: {len(test_texts)}")

# -----------------------------
# Step 5: BERT Tokenization
# -----------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 64  # Increase if symptom_text is long

def tokenize_texts(texts):
    tokens = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return tokens['input_ids'], tokens['attention_mask']

train_input_ids, train_attention_mask = tokenize_texts(train_texts)
val_input_ids, val_attention_mask = tokenize_texts(val_texts)
test_input_ids, test_attention_mask = tokenize_texts(test_texts)

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)
test_labels = torch.tensor(test_labels)

print("Tokenization complete!")
print("Train input_ids shape:", train_input_ids.shape)
print("Val input_ids shape:", val_input_ids.shape)
print("Test input_ids shape:", test_input_ids.shape)

# -----------------------------
# Step 6: Create PyTorch Datasets
# -----------------------------
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
val_dataset = TensorDataset(val_input_ids, val_attention_mask, val_labels)
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels)

# -----------------------------
# Step 7: Optional - Handle Class Imbalance in Training Set
# -----------------------------
class_counts = Counter(train_labels.tolist())
weights = 1. / torch.tensor([class_counts[label.item()] for label in train_labels], dtype=torch.float)
train_sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# -----------------------------
# Step 8: Create DataLoaders
# -----------------------------
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE)



c:\Users\Sreelakshmi\Desktop\PlantDocBot_project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Columns in dataset: ['dataset', 'plant_name', 'disease_name', 'symptom_text', 'label']
Label mapping: {'0': np.int64(0), '1': np.int64(1), '10': np.int64(2), '11': np.int64(3), '12': np.int64(4), '13': np.int64(5), '14': np.int64(6), '15': np.int64(7), '16': np.int64(8), '17': np.int64(9), '18': np.int64(10), '19': np.int64(11), '2': np.int64(12), '20': np.int64(13), '21': np.int64(14), '22': np.int64(15), '23': np.int64(16), '24': np.int64(17), '25': np.int64(18), '26': np.int64(19), '27': np.int64(20), '28': np.int64(21), '29': np.int64(22), '3': np.int64(23), '30': np.int64(24), '31': np.int64(25), '32': np.int64(26), '33': np.int64(27), '34': np.int64(28), '35': np.int64(29), '36': np.int64(30), '37': np.int64(31), '38': np.int64(32), '39': np.int64(33), '4': np.int64(34), '40': np.int64(35), '41': np.int64(36), '42': np.int64(37), '43': np.int64(38), '44': np.int64(39), '45': np.int64(40), '46': np.int64(41), '47': np.int64(42), '48': np.int64(43), '49': np.int64(44), '5': np.int6